# 1단계 · FAISS로 도서관 대출 규정에 답하기

가상의 Maple 도서관에서 **일반 도서의 대출 기간은 며칠인가?** 문서 네 개를 문단 단위로 나누고, 질문에 가까운 청크 세 개를 FAISS로 찾아 생성 모델에 함께 전달한다. 검색 순위와 모델 답변을 기록해 이후 실험의 기준으로 쓴다.

Python 3 환경에서 아래 주석의 설치 명령을 필요할 때 실행하고, 위에서부터 모든 셀을 실행한다. 생성 모델은 최초 실행 때 Hugging Face에서 내려받으며 이후에는 캐시를 사용할 수 있다. 작은 모델의 답변을 원문과 직접 대조할 수 있도록 예시 문서와 질문은 영어로 작성했다.

In [1]:
# %pip install faiss-cpu scikit-learn numpy torch transformers sentencepiece

## 문서와 청크

각 문서는 다른 상황에서 독자가 접하는 안내문이다. 빈 줄로 나뉜 문단 하나를 청크 하나로 사용한다. 청크 ID 앞부분은 원문 문서 ID이고 뒷부분은 그 안의 문단 번호다.

In [2]:
documents = {
    "handbook": """The standard loan period for library books is 14 days.

Reference books are available for reading inside the library only.""",
    "welcome-faq": """Members may borrow regular library books for 14 days before returning them.

To apply for a library card, bring a photo ID to the service desk.""",
    "checkout-slip": """A standard book loan lasts 14 days from the checkout date.

The checkout receipt lists the title and due date of each borrowed item.""",
    "returns-guide": """The outdoor book return box is next to the main entrance.

Oversized items must be returned in person at the service desk.""",
}

chunks = [
    {"id": f"{doc_id}/{number}", "text": paragraph.strip()}
    for doc_id, text in documents.items()
    for number, paragraph in enumerate(text.split("\n\n"), start=1)
    if paragraph.strip()
]
print(f"문서 {len(documents)}개 → 청크 {len(chunks)}개")
for chunk in chunks:
    print(f"{chunk['id']:16} {chunk['text']}")

문서 4개 → 청크 8개
handbook/1       The standard loan period for library books is 14 days.
handbook/2       Reference books are available for reading inside the library only.
welcome-faq/1    Members may borrow regular library books for 14 days before returning them.
welcome-faq/2    To apply for a library card, bring a photo ID to the service desk.
checkout-slip/1  A standard book loan lasts 14 days from the checkout date.
checkout-slip/2  The checkout receipt lists the title and due date of each borrowed item.
returns-guide/1  The outdoor book return box is next to the main entrance.
returns-guide/2  Oversized items must be returned in person at the service desk.


## FAISS 검색

질문과 청크를 같은 어휘의 TF-IDF 벡터로 표현한다. 두 벡터를 L2 정규화했으므로 FAISS의 내적 점수는 코사인 유사도와 같다. 아래 검색 함수는 질의와 단어가 겹치는 청크만 최대 $k$개 반환한다.

In [3]:
import faiss
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

class PassageIndex:
    def __init__(self, chunks):
        self.chunks = list(chunks)
        self.vectorizer = TfidfVectorizer(stop_words="english", dtype=np.float32, norm="l2")
        matrix = self.vectorizer.fit_transform([chunk["text"] for chunk in self.chunks])
        self.index = faiss.IndexFlatIP(matrix.shape[1])
        self.index.add(matrix.toarray())

    def search(self, question, k=3):
        if k <= 0:
            return []
        query = self.vectorizer.transform([question])
        if query.nnz == 0:
            return []
        scores, positions = self.index.search(query.toarray(), min(k, self.index.ntotal))
        return [
            {**self.chunks[position], "score": float(score)}
            for score, position in zip(scores[0], positions[0])
            if score > 0
        ]

retriever = PassageIndex(chunks)
question = "How many days is the standard loan period for library books?"
hits = retriever.search(question, k=3)

print(question)
for rank, hit in enumerate(hits, start=1):
    print(f"{rank}. [{hit['score']:.3f}] {hit['id']} · {hit['text']}")

How many days is the standard loan period for library books?
1. [0.938] handbook/1 · The standard loan period for library books is 14 days.
2. [0.417] checkout-slip/1 · A standard book loan lasts 14 days from the checkout date.
3. [0.302] welcome-faq/1 · Members may borrow regular library books for 14 days before returning them.


## 검색한 근거로 답변 생성

기본 RAG는 검색된 청크를 순서대로 하나의 프롬프트에 넣고 FLAN-T5-small을 한 번 호출한다. 같은 조건에서 다시 비교할 수 있게 모델 버전을 고정하고 샘플링을 끈다. 검색 결과가 비면 모델 호출을 건너뛴다. [모델 카드](https://huggingface.co/google/flan-t5-small)

In [4]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from transformers.utils import logging as model_logging

model_logging.disable_progress_bar()
model_id = "google/flan-t5-small"
revision = "0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab"
tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision)
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_id, revision=revision, use_safetensors=True
).eval()
torch.set_num_threads(min(torch.get_num_threads(), 4))

def make_prompt(question, passages):
    context = "\n".join(passage["text"] for passage in passages)
    return f"Answer the question using the context.\nContext: {context}\nQuestion: {question}\nAnswer:"

@torch.inference_mode()
def answer_from(question, passages):
    if not passages:
        return "근거 없음"
    inputs = tokenizer(make_prompt(question, passages), return_tensors="pt")
    output = model.generate(**inputs, do_sample=False, max_new_tokens=32)
    return tokenizer.decode(output[0], skip_special_tokens=True).strip()

print(make_prompt(question, hits))
answer = answer_from(question, hits)
print(f"\n모델 답변 → {answer}")

Answer the question using the context.
Context: The standard loan period for library books is 14 days.
A standard book loan lasts 14 days from the checkout date.
Members may borrow regular library books for 14 days before returning them.
Question: How many days is the standard loan period for library books?
Answer:

모델 답변 → 14


## 기준 결과 확인

대출 기간과 반납함 위치는 원문에서 답을 직접 확인할 수 있다. 두 질문의 생성 답변과 어휘가 전혀 겹치지 않는 질문의 처리 결과를 검사한다.

In [5]:
assert retriever.index.ntotal == len(chunks) == 8
assert len(hits) == 3 and hits[0]["id"] == "handbook/1"
assert "14" in answer or "fourteen" in answer.lower(), answer

location_question = "Where is the outdoor book return box?"
location_hits = retriever.search(location_question)
location_answer = answer_from(location_question, location_hits)
print(f"{location_question} → {location_answer} [{location_hits[0]['id']}]")
assert location_hits[0]["id"] == "returns-guide/1"
assert "main entrance" in location_answer.lower(), location_answer

unknown_question = "Is there a telescope?"
unknown_hits = retriever.search(unknown_question)
unknown_answer = answer_from(unknown_question, unknown_hits)
print(f"{unknown_question} → {unknown_answer} (검색 {len(unknown_hits)}건)")
assert not unknown_hits and unknown_answer == "근거 없음"

Where is the outdoor book return box? → next to the main entrance [returns-guide/1]
Is there a telescope? → 근거 없음 (검색 0건)
